In [ ]:
# Example 1: Parse an inline OpenAPI 2.0 (Swagger) spec from a string via BaseParser.
from prance import BaseParser

spec_yaml = '''
swagger: "2.0"
info:
  title: Demo
  version: "1.0"
paths:
  /ping:
    get:
      responses:
        "200":
          description: OK
'''

parser = BaseParser(spec_string=spec_yaml, backend='openapi-spec-validator')
print(f"version: {parser.version}")
print(f"title:   {parser.specification['info']['title']}")
print(f"paths:   {list(parser.specification['paths'].keys())}")
assert parser.specification['info']['title'] == 'Demo'

In [ ]:
# Example 2: Resolve $ref pointers in an OpenAPI 3 spec via ResolvingParser.
from prance import ResolvingParser

spec_yaml = '''
openapi: "3.0.0"
info:
  title: Refs
  version: "1.0"
paths:
  /thing:
    get:
      responses:
        "200":
          description: OK
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/Thing"
components:
  schemas:
    Thing:
      type: object
      properties:
        id: {type: string}
'''

parser = ResolvingParser(spec_string=spec_yaml, backend='openapi-spec-validator')
resolved = parser.specification['paths']['/thing']['get']['responses']['200']
schema = resolved['content']['application/json']['schema']
print(f"resolved schema keys: {sorted(schema.keys())}")
assert schema.get('type') == 'object' and 'id' in schema['properties']

In [ ]:
# Example 3: Spec format detection + content-type helpers in prance.util.formats.
from prance.util import formats

yaml_blob = 'swagger: "2.0"\ninfo:\n  title: y\n  version: "1"\n'
json_blob = '{"swagger": "2.0", "info": {"title": "j", "version": "1"}}'

yaml_parsed = formats.parse_spec(yaml_blob, 'spec.yaml')
json_parsed = formats.parse_spec(json_blob, 'spec.json')

print(f"yaml -> {yaml_parsed['info']['title']}")
print(f"json -> {json_parsed['info']['title']}")
assert yaml_parsed['swagger'] == '2.0'
assert json_parsed['swagger'] == '2.0'

In [ ]:
# Example 4: JSON pointer utilities (prance.util.path) for navigating spec dicts.
from prance.util import path as ppath

doc = {
    'paths': {
        '/users/{id}': {
            'get': {'summary': 'get one'}
        }
    }
}

ptr = ['paths', '/users/{id}', 'get']
node = ppath.path_get(doc, ptr)
print(f"node at {ptr}: {node}")

# path_set on a not-yet-existing key needs create=True (default is False -> KeyError).
ppath.path_set(doc, ptr + ['operationId'], 'getUser', create=True)
print(f"after set: {doc['paths']['/users/{id}']['get']}")
assert doc['paths']['/users/{id}']['get']['operationId'] == 'getUser'

In [ ]:
# Example 5: Validation backend selection + error surfacing on a bad spec.
from prance import BaseParser, ValidationError
from prance.util.url import ResolutionError

bad_spec = '''
openapi: "3.0.0"
info:
  title: Missing Version
paths: {}
'''

try:
    BaseParser(spec_string=bad_spec, backend='openapi-spec-validator')
except ValidationError as e:
    # The spec's `info` object has a title and no version; the validator must name the missing field.
    assert "'version' is a required property" in str(e), str(e)[:300]
    print(f"rejected as expected ({type(e).__name__}) - info.version is required")
else:
    raise AssertionError('expected prance.ValidationError on missing info.version')
print("validation-error path works")